In [4]:
import csv
import requests
from dotenv import load_dotenv
import os
import json

# Load environment variables from .env file
load_dotenv()

# Cloudflare API credentials
api_token = os.getenv('CLOUDFLARE_API_TOKEN')
account_id = os.getenv('CLOUDFLARE_ACCOUNT_ID')

# Function to retrieve all Cloudflare page projects
def get_all_cloudflare_projects():
    # check if projects.json file exists and return the projects from the file
    if os.path.exists('projects.json'):
        with open('projects.json', 'r') as f:
            return json.load(f)
        
    url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/pages/projects"
    headers = {
        'Authorization': f'Bearer {api_token}',
        'Content-Type': 'application/json',
    }
    
    projects = []
    page = 1

    while True:
        params = {'page': page}
        response = requests.get(url, headers=headers, params=params)
        print(response.content)
        if response.status_code == 200:
            result = response.json()['result']
            if not result:
                break
            projects.extend(result)
            page += 1
        else:
            print(f"Failed to retrieve projects: {response.content}")
            break
    
    # save the projects to a file
    with open('projects.json', 'w') as f:
        f.write(json.dumps(projects, indent=4))
    return projects

# Function to find a Cloudflare page project by domain
def find_cloudflare_project(domain, projects):
    page_name = domain.replace('.', '-')
    print(f"Searching for project with name {page_name}")  
    
    for project in projects:
        if project['name'] == page_name:
            return project['name']
    
    print(f"No project found for {domain}")
    return None

# Function to deploy a Cloudflare page
def deploy_cloudflare_page(project_id):
    url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/pages/projects/{project_id}/deployments"
    print(url)
    headers = {
        'Authorization': f'Bearer {api_token}',
        'Content-Type': 'application/json',
    }
    response = requests.post(url, headers=headers)
    print(response.content)
    if response.status_code == 200:
        print(f"Deployment triggered successfully for project ID {project_id}")
    else:
        print(f"Failed to deploy project ID {project_id}: {response.content}")

# Read the CSV file and process each domain
def main():
    print("Starting...")
    projects = get_all_cloudflare_projects()
    # id = find_cloudflare_project('money.allwomenstalk.com',projects)
    # print(id)
    # deploy_cloudflare_page(id)
    with open('hosts.csv', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            domain = row['host']
            if 'allwomenstalk.com' in domain:
                project_id = find_cloudflare_project(domain, projects)
                if project_id:
                    deploy_cloudflare_page(project_id)

if __name__ == "__main__":
    main()






Starting...
Searching for project with name money-allwomenstalk-com
money-allwomenstalk-com
https://api.cloudflare.com/client/v4/accounts/cbe44b125e21f7aada0eefba2df8fc30/pages/projects/money-allwomenstalk-com/deployments
b'{\n  "result": {\n    "id": "310d260a-e6ed-4334-8d57-332a629b92e4",\n    "short_id": "310d260a",\n    "project_id": "b5d138ae-73ca-4c08-9e20-afcc454be534",\n    "project_name": "money-allwomenstalk-com",\n    "environment": "production",\n    "url": "https://310d260a.money-allwomenstalk-com.pages.dev",\n    "created_on": "2024-07-05T09:07:42.436775Z",\n    "modified_on": "2024-07-05T09:07:42.436775Z",\n    "latest_stage": {\n      "name": "queued",\n      "started_on": null,\n      "ended_on": null,\n      "status": "idle"\n    },\n    "deployment_trigger": {\n      "type": "ad_hoc",\n      "metadata": {\n        "branch": "main",\n        "commit_hash": "5f5386799f4180e8f49395773fd714435687da7c",\n        "commit_message": "Update post awesome-marketing-ideas-for-b